<a href="https://colab.research.google.com/github/JuanZapa7a/Medical-Image-Processing/blob/main/PIM_Challenge/PIM_Challenge_Student_Practice_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UPCT Medical Image Segmentation Challenge 2026-27
## Practice 7

**Course:** Medical Image Processing (521104007)

**Professor:** Juan Zapata

> **New** content for this practice. Copy the cells below and paste them **at the end** of your own notebook (the one you started in Practice 6) — do not repeat the previous practices, you already have them done there.

## Session Guide (2 hours per session)
| Practice | Dates (Group A / B) | Session Objective | Visual Checkpoint |
|----------|----------------------|-------------------|-------------------|
| **P6** | 28 Oct - 2 Nov | EDA, Dataset and RLE format | 6 images with masks + RLE OK |
| ▶ **P7** | 9-11 Nov | U-Net baseline and 1st Submission | Loss plots + Kaggle Submission |
| **P8** | 16-18 Nov | Data Augmentation and improvement | Baseline vs Augmented comparison |
| **P9** | 23-25 Nov | Inference, Threshold and Errors | 5 normal images + 2 error cases |
| **P10** | 30 Nov-2 Dec | TTA, Final Submission and Defense | Best Dice Score + Oral Defense |

> **Golden Rule:** According to Art. 7.5 of the UPCT Evaluation Regulations, attendance and checkpoint validation in the classroom is mandatory to pass the practice.


# Practice 7: Baseline Model (U-Net) and 1st Submission
## Single session (9 Nov Group A / 11 Nov Group B)

### Session objectives:
1. Create a PyTorch `Dataset` and `DataLoader` to load the images and masks.
2. Define the **U-Net** architecture and the loss function (**Dice Loss + BCE**).
3. Train the model for a few epochs (baseline).
4. Run inference on the test set and generate the `submission.csv` file.

> **CHECKPOINT P7:** Show the professor the decreasing loss plots and the screenshot of your first submission on the Kaggle Leaderboard.

## Block 7.1: Dataset and DataLoader in PyTorch
### Why is a list of arrays not enough?

So far you have worked with images loaded "by hand" (a standalone `cv2.imread`, a `for` loop). To train a neural network you need something more:

| Without `Dataset`/`DataLoader` | With `Dataset`/`DataLoader` |
|---|---|
| Manual loop loading everything into RAM | **On-demand** loading, one sample at a time |
| Batches built by hand | Groups samples into **batches** automatically |
| Manual shuffling (or none) | `shuffle=True` reorders every epoch |
| A single process reading disk | `num_workers` loads in **parallel** |

With 546 images this does not seem critical, but it is the same pattern you will use with datasets of millions of images: **PyTorch does not change its API**, only the size of the dataset changes.

### The abstraction: two pieces with different responsibilities

```
DataFrame (index)         Dataset                    DataLoader
class, image_path,   →    __getitem__(idx)    →      joins N samples
mask_path, filename        returns 1 sample          into a batch
                            (image, mask)           (B, C, H, W)
```

| | `Dataset` | `DataLoader` |
|---|---|---|
| **Answers** | "How do I get sample `i`?" | "How do I group and serve the samples?" |
| **Key methods** | `__len__`, `__getitem__` | — (it is a wrapper) |
| **You decide** | How to load, preprocess and return 1 sample | `batch_size`, `shuffle`, `num_workers` |

> **Key idea:** the `Dataset` knows nothing about batches. It only knows how to return **one** sample given an index. All the logic of grouping, shuffling and parallelizing is the `DataLoader`'s responsibility.

### The segmentation challenge: (image, mask) pairs

In classification, `__getitem__` returns `(image, label)` — the label is a number, it does not need to be transformed.

In **segmentation** you return `(image, mask)`, and here there is a trap:

- **Geometric** transformations (resize, flip, rotation) must be applied **identically** to the image and the mask — if you rotate the image 15° and you do not rotate the mask, they are no longer aligned.
- **Photometric** transformations (normalization, brightness, contrast, noise) only make sense on the **image** — the mask must remain binary (0/1), never "brighten it" or "add noise to it".

> **Question to think about:** if in Practice 8 you rotate the image 15° inside `__getitem__` but you forget to rotate the mask, what Dice Score would you expect to get? High, low, or would it depend on the image?

### Normalization: the hidden trap of pretrained encoders

A common normalization is dividing by 255 to bring the pixels to `[0, 1]`. It is correct... **but incomplete** for our case.

Our model (Practice 7) uses a **ResNet34 encoder pretrained on ImageNet** (`encoder_weights="imagenet"`). That encoder was trained with images normalized with the **ImageNet mean and standard deviation**, not with plain `[0, 1]`:

| Channel | Mean | Std |
|-------|-------|--------------------|
| R | 0.485 | 0.229 |
| G | 0.456 | 0.224 |
| B | 0.406 | 0.225 |

```python
image = image.astype(np.float32) / 255.0
image = (image - mean) / std   # mean, std per channel, above
```

If you feed the pretrained encoder with data in `[0, 1]` without centering, the network "sees" an input distribution different from the one it learned — training will still work, but you will start from a worse position and converge more slowly (and with Data Augmentation, Albumentations does this normalization automatically for you via `Normalize()`, so from Practice 8 onwards you will not write it yourself).

### Axis convention: `(H, W, C)` vs `(C, H, W)`

- `cv2` / `numpy` represent an image as `(Height, Width, Channels)`.
- PyTorch expects `(Channels, Height, Width)` for everything that enters a `Conv2d` layer.

```python
image_tensor = torch.tensor(image).permute(2, 0, 1)   # (H,W,C) → (C,H,W)
```

The mask is a special case: it is loaded as `(H, W)` (a single channel, without an explicit channel dimension). Before feeding it to the model it needs a channel dimension: `(1, H, W)` — that is why you will see `.unsqueeze(0)`.

### Train/Val Split: is random always enough?

`train_test_split(df, test_size=0.2)` distributes rows at random. But remember the "hidden challenge" of Practice 6: the dataset has only 133 `normal` images out of 780. A purely random split does not guarantee that this proportion is maintained in the validation set.

> **Question to think about:** what would happen to your `best_threshold` (Practice 9) if by bad luck the split left almost no `normal` image in validation?

*(Hint for those who want to go further: `train_test_split(..., stratify=df['class'])` keeps the class proportions in train and val.)*

### The `DataLoader` parameters

| Parameter | What it does | Practical rule |
|---|---|---|
| `batch_size` | Number of samples per step | Higher = faster, more GPU memory |
| `shuffle` | Reorders the samples every epoch | `True` in train, `False` in val |
| `num_workers` | Loads in parallel from disk | In Colab, 2 workers is usually enough |

> `shuffle=False` in val is not an oversight: this way you always compare the same samples in the same order between epochs.

### Quick summary

| Concept | Main idea |
|----------|----------------|
| `Dataset` | Returns 1 sample given an index (`__getitem__`) |
| `DataLoader` | Groups into batches, shuffles and parallelizes loading |
| Image vs Mask | Same geometric transf.; normalization only on the image |
| Normalization | `/255` is not enough with pretrained encoders (ImageNet mean/std is needed) |
| `(H,W,C)` → `(C,H,W)` | numpy/cv2 convention vs PyTorch — use `permute` |
| Train/val split | Simple random split can unbalance minority classes (`normal`) |

### References

1. **PyTorch Docs.** *`torch.utils.data.Dataset` and `DataLoader`.* pytorch.org/docs/stable/data.html
2. **Deng, J., et al. (2009).** *ImageNet: A Large-Scale Hierarchical Image Database.* CVPR. *(source of the mean/std normalization statistics)*
3. **He, K., et al. (2016).** *Deep Residual Learning for Image Recognition.* CVPR. *(ResNet, the encoder you will use in the U-Net)*

> Next step: now that you know what each piece must do, implement `BUSIDataset` and its `DataLoader`s in Task 7.1.

## Task 7.1: Dataset and DataLoader
To train in PyTorch, we need to package our data.
1. Create a `BUSIDataset` class that inherits from `torch.utils.data.Dataset`.
2. In `__getitem__`, load the image (RGB) and the mask (grayscale).
3. Resize both to `IMG_SIZE` (e.g. 256x256).
4. Normalize the image (divide by 255.0) and binarize the mask (threshold > 127).
5. Return the image as a `(C, H, W)` tensor and the mask as a `(1, H, W)` tensor.
6. Create the `DataLoader`s for train and validation (use 20% for val).

> **Hint:** Use `torch.tensor(img).permute(2, 0, 1)` to change the image format.

In [ ]:
# ============================================================
# TASK 7.1: DATASET AND DATALOADER
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np

# Hyperparameters
IMG_SIZE = 256
BATCH_SIZE = 16

# WRITE YOUR CODE HERE
class BUSIDataset(Dataset):
    def __init__(self, dataframe, img_size=IMG_SIZE):
        self.df = dataframe.reset_index(drop=True)
        self.img_size = img_size

    def __len__(self):
        # Your code here
        pass

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. Load image and mask
        # Your code here

        # 2. Resize to IMG_SIZE
        # Your code here

        # 3. Normalize image and binarize mask
        # Your code here

        # 4. Convert to PyTorch tensors
        # Your code here

        return image_tensor, mask_tensor

# Split the dataset (80% train, 20% val)
# Your code here to create train_df and val_df from df_train

# Create Datasets and DataLoaders
# Your code here

## Block 7.2: U-Net Architecture and Loss Function
### Classification vs Detection vs Segmentation

| Task | Question it answers | Output |
|-------|----------------------|--------|
| **Classification** | Is there a tumor? | Label: "benign" / "malignant" |
| **Detection** | Where is the tumor? | Bounding box |
| **Segmentation** | Which pixels are tumor? | Pixel-by-pixel mask |

### Why segmentation and not classification?
- In medicine, the **size and shape** of the tumor matter (staging)
- It allows computing **volume**, **irregular borders**, **invasion**
- It is the basis of **radiomics** and assisted diagnosis

### The specific challenge: Breast Ultrasound (BUS)
- **Noisy** and **low-contrast** images
- **Small** tumors (sometimes < 5% of the image)
- Artifacts: acoustic shadows, reverberations
- **"Normal" class**: the model must learn NOT to detect anything

### What is Semantic Segmentation?

Before talking about U-Net, let us recall what we are doing:

| Task | Input | Output |
|-------|---------|--------|
| **Classification** | Image | One label ("benign", "malignant") |
| **Detection** | Image | Bounding boxes + labels |
| **Segmentation** | Image | **Pixel-by-pixel mask** |

In our case, the output is a **binary mask** of the same size as the input image, where each pixel has a value:
- `1` → belongs to the tumor
- `0` → is healthy tissue (background)

### U-Net Architecture

#### Origin

Proposed by **Ronneberger et al. (2015)** in the paper *"U-Net: Convolutional Networks for Biomedical Image Segmentation"*. It was designed specifically for **biomedical segmentation** with few training data.

#### Key Idea: "U" Shape

The architecture has a **U** shape because it combines two paths:

```
Input → [ENCODER] → [BOTTLENECK] → [DECODER] → Output
   ↓          ↓              ↓             ↓
   572x572   28x28          4x4           572x572
```

### Encoder (Contracting Path)

The encoder is similar to a classic classification CNN. Its job is to **extract features** and reduce the spatial resolution:

```
Input (256x256x3)
    │
    ▼
[Conv 3x3] → [Conv 3x3] → [MaxPool 2x2]  → 128x128
    │
    ▼
[Conv 3x3] → [Conv 3x3] → [MaxPool 2x2]  → 64x64
    │
    ▼
[Conv 3x3] → [Conv 3x3] → [MaxPool 2x2]  → 32x32
    │
    ▼
[Conv 3x3] → [Conv 3x3] → [MaxPool 2x2]  → 16x16
    │
    ▼
[Conv 3x3] → [Conv 3x3]                  → 8x8  (Bottleneck)
```

**What does the encoder do?**
- Extracts **low-level** features (edges, textures) in the first layers
- Extracts **high-level** features (shapes, complex patterns) in the deep layers
- **Reduces** spatial resolution but **increases** channels (more semantic information)

### Bottleneck

It is the deepest point of the network. Here the image has been greatly reduced spatially, but it contains the **most abstract semantic information**.

### Decoder (Expansive Path)

The decoder **reconstructs** the spatial resolution pixel by pixel:

```
Bottleneck (8x8)
    │
    ▼
[UpConv 2x2] → [Conv 3x3] → [Conv 3x3]   → 16x16
    │
    ▼
[UpConv 2x2] → [Conv 3x3] → [Conv 3x3]   → 32x32
    │
    ▼
[UpConv 2x2] → [Conv 3x3] → [Conv 3x3]   → 64x64
    │
    ▼
[UpConv 2x2] → [Conv 3x3] → [Conv 3x3]   → 128x128
    │
    ▼
[Conv 1x1] + Sigmoid                         → 256x256x1 (Mask)
```

### Skip Connections: The "Secret" of U-Net

Here is the **key innovation**: the **skip connections** that link the encoder with the decoder:

```
Encoder                    Decoder
   │                          │
   ├──[feat1]──────────────►[+ merge]──►
   │                          │
   ├──[feat2]──────────────►[+ merge]──►
   │                          │
   ├──[feat3]──────────────►[+ merge]──►
   │                          │
   └──[feat4]──────────────►[+ merge]──►
                              │
                           [Output]
```

**Why are they so important?**

| Without Skip Connections | With Skip Connections |
|---------------------|---------------------|
| Only semantic information | Semantic information **+** precise localization |
| Blurry borders | Sharp borders |
| Loses fine details | Recovers fine details |

> **Analogy:** Imagine that the decoder is a painter who has to reconstruct a painting. The encoder gives him the "general idea" (what is painted), but the skip connections pass him the "original sketches" so he can paint with precision.

### Loss Function: Dice Loss + BCE

In our notebook we use a **combined loss function**:

```python
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum()
        dice_loss = 1 - (2. * intersection + self.smooth) / \
                    (probs.sum() + targets.sum() + self.smooth)

        return bce_loss + dice_loss
```

### 1. Binary Cross Entropy (BCE)

It is the classic loss for binary classification, applied **pixel by pixel**:

$$\text{BCE} = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i) \right]$$

Where:
- $y_i$ = real value of pixel $i$ (0 or 1)
- $\hat{y}_i$ = model prediction for pixel $i$
- $N$ = total number of pixels

**Intuition:**
- If the real pixel is `1` and you predict `0.9` → **low** penalty
- If the real pixel is `1` and you predict `0.1` → **high** penalty

**BCE problem in segmentation:**
- Medical images have **extreme class imbalance**
- Example: in a 256×256 = 65,536 pixel image, maybe only 500 are tumor (~0.7%)
- BCE treats all pixels equally → the model can learn to predict everything as "background" and have good accuracy

### 2. Dice Loss

Based on the **Dice Coefficient** (or *F1 Score*):

$$\text{Dice} = \frac{2 \cdot |A \cap B|}{|A| + |B|}$$

Where:
- $A$ = pixels predicted as tumor
- $B$ = real tumor pixels
- $|A \cap B|$ = intersection (correctly classified pixels)

The **Dice Loss** is simply:

$$\text{Dice Loss} = 1 - \text{Dice} = 1 - \frac{2 \cdot |A \cap B| + \epsilon}{|A| + |B| + \epsilon}$$

The term $\epsilon$ (smooth) avoids **division by zero** when both masks are empty.

**Visual intuition:**

```
    A (Prediction)          B (Ground Truth)        A ∩ B (Intersection)
    ┌─────────┐             ┌─────────┐             ┌─────────┐
    │  ████   │             │   ████  │             │   ███   │
    │  ████   │             │   ████  │             │   ███   │
    │  ████   │             │   ████  │             │   ███   │
    └─────────┘             └─────────┘             └─────────┘

    Dice = 2·(intersection) / (|A| + |B|)
```

**Advantages of Dice Loss:**
- **Robust to class imbalance**
- Directly optimizes the **metric that matters to us** (Dice Score)
- Works well with **small objects** (small tumors)

**Disadvantages of Dice Loss:**
- Unstable at the beginning of training (noisy gradients)
- Does not consider the **spatial location** of each individual pixel

### Why combine them?

| Loss | Strength | Weakness |
|---------|-----------|-----------|
| **BCE** | Stable gradients, considers each pixel | Sensitive to imbalance |
| **Dice** | Robust to imbalance | Unstable gradients at the start |

**Combining them we get the best of both worlds:**

$$\mathcal{L}_{total} = \mathcal{L}_{BCE} + \mathcal{L}_{Dice}$$

- **BCE** provides stable gradients from the first batch
- **Dice** pushes the model to maximize the segmentation metric
- Together they converge faster and to better results

## Application to Our Challenge

In the *UPCT Medical Image Segmentation Challenge*:

```python
# U-Net model with pre-trained ResNet34 encoder
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

# Combined loss
criterion = DiceBCELoss(smooth=1.0)

# Adam optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)
```

### Complete flow:

```
Image (256×256×3)
    │
    ▼
[ResNet34 Encoder] → Extracts hierarchical features
    │
    ▼
[Decoder] → Reconstructs mask with skip connections
    │
    ▼
Logits (256×256×1)
    │
    ▼
[Sigmoid] → Probabilities [0, 1]
    │
    ▼
[Threshold 0.5] → Binary mask
    │
    ▼
[DiceBCELoss] → Compare with ground truth
    │
    ▼
[Backpropagation] → Update weights
```


### References

1. **Ronneberger, O., Fischer, P., & Brox, T. (2015).** *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI.
2. **Sudre, C. H., et al. (2017).** *Generalised Dice overlap as a deep learning loss function for highly unbalanced segmentations.* Deep Learning in Medical Image Analysis.
3. **Milletari, F., Navab, N., & Ahmadian, S. A. (2016).** *V-Net: Fully convolutional neural networks for volumetric medical image segmentation.* 3DV.


### Quick Summary

| Concept | Main Idea |
|----------|----------------|
| **U-Net** | Encoder + Decoder + Skip Connections |
| **Encoder** | Extracts features, reduces resolution |
| **Decoder** | Reconstructs the mask pixel by pixel |
| **Skip Connections** | Preserve high-level spatial information |
| **BCE** | Pixel-by-pixel loss, stable but sensitive to imbalance |
| **Dice Loss** | Robust to imbalance, optimizes the final metric |
| **Dice + BCE** | The best of both worlds |

> **Next step:** In practice, we will implement U-Net with `segmentation_models_pytorch` and train with the combined loss. Let's code!


## Task 7.2: U-Net Model and Loss Function
We are going to use the `segmentation_models_pytorch` (smp) library to create a U-Net with a pre-trained backbone (ResNet34).
1. Define the `smp.Unet` model with `encoder_weights='imagenet'` and `activation=None` (important for the loss since we use BCEWithLogitsLoss).
2. Define a custom loss class `DiceBCELoss` that combines `BinaryCrossEntropyWithLogitsLoss` and `DiceLoss`.
3. Configure the `Adam` optimizer.

In [ ]:
# ============================================================
# TASK 7.2: MODEL AND LOSS
# ============================================================
# Install the libraries needed for segmentation
!pip install -q segmentation_models_pytorch timm

import segmentation_models_pytorch as smp
import torch.nn as nn
import torch.optim as optim

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# 1. Define the U-Net model
# Your code here (use smp.Unet)
model = ...
model = model.to(DEVICE)

# 2. Define the combined loss function (Dice + BCE)
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        # Your code here: compute BCE and Dice Loss and add them
        pass

criterion = DiceBCELoss().to(DEVICE)

# 3. Optimizer
# Your code here (use optim.Adam with lr=1e-4)
optimizer = ...

## Block 7.3: The Training Loop in PyTorch
### Anatomy of a training step

  Each batch goes through the same five lines, always in the same order. It is worth memorizing the order because the three most common PyTorch bugs come from breaking it:

  ```python
  optimizer.zero_grad()   # 1. Zero the gradients from the previous step
  logits = model(images)  # 2. Forward pass
  loss = criterion(logits, masks)  # 3. Compute the loss
  loss.backward()         # 4. Backward pass (computes gradients)
  optimizer.step()        # 5. Update the weights with those gradients
  ```

  | Step | What it does | What happens if you forget or omit it |
  |------|----------|----------------------------------|
  | `optimizer.zero_grad()` | Clears the gradients accumulated from the previous batch | Gradients **add up** between batches; the model learns badly and erratically |
  | `model(images)` | Computes the prediction (logits, without sigmoid yet) | — |
  | `criterion(logits, masks)` | Compares prediction vs. real mask | — |
  | `loss.backward()` | Computes how much each weight contributes to the error (gradients) | Without this, `optimizer.step()` has nothing to update |
  | `optimizer.step()` | Moves the weights in the direction that reduces the error | Without this, the model never changes even if you compute gradients |


### Epochs, batches and steps

Three words that are easily confused:

| Term | Definition | In our case |
|---------|------------|------------------|
| **Step** | One weight update = one processed batch | `optimizer.step()` once |
| **Batch** | A group of `batch_size` samples | 16 images (`BATCH_SIZE = 16`) |
| **Epoch** | One full pass over the whole `train_loader` | `len(train_loader)` steps |

`len(train_loader)` is not the number of images, it is the number of **batches**: approximately `num_samples / batch_size`. If you have 437 train samples and `batch_size=16`, each epoch is 28 steps, not 437.

### The difference between `model.train()` and `model.eval()`

Your encoder (ResNet34) uses **BatchNorm** layers. These layers behave differently depending on the mode:

| Mode | BatchNorm uses... | When to activate it |
|------|-------------------|--------------------|
| `model.train()` | Statistics of the current batch (mean/variance) | During training |
| `model.eval()` | Statistics accumulated over the whole training | During validation and inference |

If you evaluate with the model in `train()` mode, the result depends on which other images are in that validation batch — two runs with the same model can give different Dice Scores. That is why, before each validation phase, you must call `model.eval()`, and before training again, `model.train()` once more.

### How `torch.no_grad()` behaves during validation

When training, PyTorch builds a computation graph in order to compute gradients with `backward()`. That graph consumes memory and time.

In validation you are **not going to call `backward()`**, so that graph is wasted work. `torch.no_grad()` tells PyTorch not to build it:

```python
model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        ...
```

Without `torch.no_grad()`, the validation loop is still *correct* (the numerical result does not change), but it is slower and can exhaust GPU memory on large datasets.


### Averaging the metric per epoch

Inside the loop you accumulate the loss and the Dice **per batch**:

```python
epoch_train_loss += loss.item()
```

When all the batches of the epoch finish, you have to divide by the number of batches to get the real average:

```python
epoch_train_loss /= len(train_loader)
```

If you forget this division, the number you print is not a per-batch loss but an **accumulated sum**, which grows with each batch within the same epoch and is not comparable between epochs with a different number of batches.

### Why save Loss and Dice, not just one

The loss function (`DiceBCELoss`) is what **optimizes** the model, but it is not necessarily the metric you care about as a clinical or competition result. Saving both allows you to detect situations where they diverge:

| Situation | Loss | Dice | Interpretation |
|-----------|------|------|-----------------|
| Normal case | Low | Goes up | The model improves in both senses |
| Deceptive plateau | Decreases little by little | Stagnates | The loss keeps optimizing but does not translate into better real segmentation |
| Train/val divergence | Train down, Val up | Train up, Val down | Sign of overfitting (you will see it in detail in Practice 8) |

> **Note:** Kaggle does not evaluate your Loss, it evaluates the Dice Score on the test set. The Loss is only the signal you use internally to train.

### Keep the best model, not the last one (checkpointing)

The previous loop trains all `NUM_EPOCHS` epochs and, when it finishes, `model` contains the weights of the **last** epoch — not necessarily the best one. If the Val Dice goes up and down from one epoch to another (something normal, not a bug), the last epoch can be worse than a previously seen one.

The solution is to save a copy of the weights every time the Val Dice improves, and at the end of the loop restore that copy:

```python
if epoch_val_dice > best_val_dice:
    best_val_dice = epoch_val_dice
    torch.save(model.state_dict(), 'best_model.pth')
```

`model.state_dict()` is a dictionary with all the weights of the model at that moment; `torch.save` stores it on disk. At the end of the loop, `model.load_state_dict(torch.load('best_model.pth'))` overwrites the current weights of `model` with the ones you saved — so any later code (Task 7.4, Practice 9, Practice 10) that uses `model` automatically uses the best checkpoint, without needing to change anything else.

> **Key idea:** saving only the number (`max(val_dices)`) is useless if you do not also save the weights that produced it — the number without the weights is just a statistical curiosity.

### Why Google Drive and not the Colab disk

These practices are done on different days. Every day you open Colab it is a **new runtime**: the local disk (`/content/...`) is completely wiped, so a checkpoint saved there only survives within the session where you created it. If you saved the checkpoint only locally, you would have to retrain the model from scratch every day just to have `model` ready again, before even being able to start the new task of the day.

Google Drive does persist between sessions. The `DRIVE_FOLDER` folder was already mounted in Practice 6 to download the dataset — we reuse that same folder to also save the checkpoints, in a `checkpoints/` subfolder.

With this, each training cell follows this logic:

| Situation | What the cell does |
|-----------|---------------------|
| No checkpoint exists in Drive (first time) | Trains the `NUM_EPOCHS` epochs normally and, when finished, saves weights + history (Loss/Dice) in Drive |
| A checkpoint already exists in Drive (session from a previous day) | Loads the saved weights and history directly — **does not retrain** |

> **Key idea:** the checkpoint in Drive is not only the model weights — it also includes the history of `train_losses`, `val_losses`, `train_dices` and `val_dices`, because the plots and comparisons of later tasks (for example, Task 8.4) need those lists even if the model was trained days ago.

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| `zero_grad → forward → loss → backward → step` | Fixed order; skipping `zero_grad` accumulates gradients between batches |
| Epoch vs batch vs step | One epoch = `len(train_loader)` steps, not the number of images |
| `model.train()` / `model.eval()` | Changes BatchNorm behavior; you must alternate between phases |
| `torch.no_grad()` | Avoids building the gradient graph in validation (memory and speed) |
| Accumulate and divide | Sum per batch, divide by `len(loader)` at the end of the epoch |
| Loss vs Dice | The Loss optimizes, the Dice is what is actually evaluated |
| Checkpointing | Save the weights of the best Val Dice and reload them at the end; otherwise you keep the ones from the last epoch |
| Checkpoint in Drive | Persists between sessions on different days; the cell checks if it already exists and avoids retraining if so |


## Task 7.3: Training Loop
Time to train.
1. Create a loop for `NUM_EPOCHS` (start with 5 or 10 for the class, it is so you can check that everything works well).
2. In each epoch, iterate over the `train_loader`: forward pass, compute loss, backward pass and `optimizer.step()`.
3. At the end of each epoch, evaluate the model on the `val_loader` (in `model.eval()` and `torch.no_grad()` mode).
4. Save the train and val loss in lists to plot at the end. It is interesting to observe the dice both in training and in validation since it is what Kaggle will score.
5. Save the model weights (`torch.save(model.state_dict(), ...)`) every time the validation Val Dice improves with respect to previous epochs, and when the loop finishes load those weights into `model` with `load_state_dict`. This way you keep the best model, not the one from the last epoch.
6. Save the final checkpoint (weights + history) in the Google Drive folder where the dataset is (`DRIVE_FOLDER`, mounted in P6), not only on the local Colab disk — so it persists between sessions on different days. At the beginning of the cell, check if a checkpoint from a previous session already exists in Drive: if it does, load it directly (together with the Loss/Dice history) and do not train again.

> **CHECKPOINT P7.1:** Show the professor the decreasing loss plots (training and validation) and the growing Dice Score (training and validation)

In [ ]:
# ============================================================
# TASK 7.3: TRAINING
# ============================================================
from pathlib import Path

NUM_EPOCHS = 10 # We start with a few epochs to see results quickly

# Checkpoints folder in Google Drive (persists between different sessions/days)
CHECKPOINT_DIR = Path(DRIVE_FOLDER) / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_PATH = CHECKPOINT_DIR / 'baseline_checkpoint.pth'

train_losses = []
val_losses = []
train_dices = []
val_dices = []

if DRIVE_CHECKPOINT_PATH.exists():
    # This model was already trained in a previous session: load it instead of retraining
    print(f"Checkpoint found in Google Drive: {DRIVE_CHECKPOINT_PATH}")
    # Your code here: torch.load(DRIVE_CHECKPOINT_PATH) and restore model, train_losses,
    # val_losses, train_dices, val_dices and best_val_dice from the checkpoint

else:
    # 1. Variables to keep the best model (not the one from the last epoch)
    # Your code here: best_val_dice = -1 and BEST_MODEL_PATH = 'best_model_baseline.pth'

    print("Starting training...")
    # 2. Create training loop
    for epoch in range(NUM_EPOCHS):
        # 3. Training Phase (model.train())
        # Your code here to iterate train_loader and update weights

        # 4. Validation Phase (model.eval() and torch.no_grad())
        # Your code here to iterate val_loader and compute loss without gradients

        # 5. save in lists to plot later
        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)

        # 6. Save local checkpoint if this epoch's Val Dice is the best so far
        # Your code here: compare epoch_val_dice with best_val_dice and, if it improves,
        # update best_val_dice and do torch.save(model.state_dict(), BEST_MODEL_PATH)

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {epoch_train_loss:.4f} - Val Loss: {epoch_val_loss:.4f}")

    # 7. Load the weights of the best local model (do not keep the ones from the last epoch)
    # Your code here: model.load_state_dict(torch.load(BEST_MODEL_PATH))

    # 8. Save the final checkpoint (weights + history) in Google Drive so you do not retrain in future sessions
    # Your code here: torch.save({...}, DRIVE_CHECKPOINT_PATH)

# Plot results
import matplotlib.pyplot as plt
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

## Block 7.4: From Training to Submission
### What changes (and what does not) when moving to inference

Training and predicting on test share almost all the code, with three key differences:

| | Training / Validation | Inference on test |
|---|---|---|
| Model mode | `model.train()` in train, `model.eval()` in val | `model.eval()` always |
| Gradients | Computed in train (`loss.backward()`) | Never (`torch.no_grad()`) |
| Are there real `masks`? | Yes, to compute loss and Dice | No — the test set does not come with masks |
| Output that matters | A number (loss, Dice) to compare models | A binary mask to send to Kaggle |

> **Note:** since there is no real mask in test, you cannot compute your Dice Score locally on these images. The only way to know how well your model does there is to submit and look at the Leaderboard.

### The preprocessing must be identical to training

This is the point where it is easiest to introduce a silent bug. The model learned from images preprocessed in a specific way (Block 7.1: resize to `IMG_SIZE`, normalization with ImageNet mean/std). If in inference you preprocess differently — even if it is "only" skipping the ImageNet normalization and staying at `/255.0` — the model receives data with a distribution different from what it saw in training, and its performance drops without any error being raised.

```python
# It must be the SAME preprocessing as in BUSIDataset (Block 7.1), step by step:
img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img_norm = img_resized.astype(np.float32) / 255.0
img_norm = (img_norm - mean) / std          # the same ImageNet mean/std
img_tensor = torch.tensor(img_norm).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
```

> **Question to think about:** if you train with ImageNet normalization but in inference you only divide by 255, will the model give an error, or simply worse predictions without warning? Why is the second case more dangerous?

### The predicted mask does not have the original size

The model always works at `IMG_SIZE x IMG_SIZE` (for example 256x256), but each test image has its own original size. Kaggle expects the mask at the original resolution of the image, so you have to undo the resize before encoding in RLE:

```
Original image (H0, W0)
        │  resize to IMG_SIZE
        ▼
Model → mask (IMG_SIZE, IMG_SIZE)
        │  resize back to (H0, W0)
        ▼
Mask at original size → mask_to_rle()
```

Save `original_h, original_w` **before** resizing the input image — it is easy to forget it and keep only the already rescaled size.

### From probability to binary mask

The model does not directly return a mask — it returns **logits**, which must be converted in two steps:

| Step | Operation | Result |
|------|-----------|-----------|
| 1. Logits → probability | `torch.sigmoid(logits)` | Continuous values between 0 and 1 |
| 2. Probability → binary mask | `(probs > threshold)` | Values 0 or 1 |

Here we use `threshold = 0.5` because it is the reasonable default value, but there is no guarantee that it is the best cutoff for this problem — in Practice 9 you will look for the `best_threshold` specific to your model.

### Reuse `mask_to_rle`: why not rewrite it

The `mask_to_rle` function was already implemented in Practice 6. The idea of a well-organized `Dataset`/pipeline is precisely this: each piece (data loading, model, RLE encoding) is written once and reused in all the following practices. If you have to copy and paste the same RLE logic in every practice, it is a sign that you should save it in a utilities cell at the beginning of the notebook instead of repeating it.

### The `submission.csv` format

Kaggle validates the submission by comparing exact columns against `sample_submission.csv`:

| Column | Content |
|---------|-----------|
| `Id` | Image file name (must match exactly the one in test) |
| `Expected` | The RLE string of the predicted mask for that image |

A row with a misspelled `Id`, or an RLE with indices that do not match the declared dimensions, makes Kaggle reject or score poorly that row — it is worth comparing your `submission.csv` against `sample_submission.csv` (same number of rows, same `Id`s) before uploading it.

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| Test set | It has no masks; you will only know your real Dice when you upload to Kaggle |
| Preprocessing | Must be identical to training (resize + normalization), or the model performs worse without an error |
| Resize the mask | The model predicts at `IMG_SIZE`; you must return the mask to the original size before RLE |
| Logits → mask | `sigmoid()` gives probabilities, a `threshold` turns them into a binary mask |
| `mask_to_rle` | Reuse the one from Practice 6, do not rewrite it |
| `submission.csv` | Columns `Id` and `Expected`; compare against `sample_submission.csv` before uploading |




## Task 7.4: Inference and Submission to Kaggle
The model is trained! Now you have to predict on the test images.
1. Iterate over the images in the `test/images` folder.
2. Pass the image through the model (remember to preprocess it the same as in train).
3. Apply `torch.sigmoid()` to the output and threshold it (e.g. > 0.5) to obtain the binary mask.
4. Resize the mask to the original size of the image.
5. Convert the mask to RLE format using the `mask_to_rle` function from Practice 6.
6. Save the results in a DataFrame and export it as `submission.csv`.

> **CHECKPOINT P7.2:** Download the `submission.csv` and upload it to Kaggle. Show your position on the Leaderboard!

In [ ]:
# ============================================================
# TASK 7.4: INFERENCE AND SUBMISSION
# ============================================================
import os
import pandas as pd

# Make sure you have the mask_to_rle function defined (copy it from P6 if necessary)
# def mask_to_rle(mask): ...

test_img_dir = DATA_DIR / 'test' / 'images'
test_images = list(test_img_dir.glob('*.png'))

results = []

print("Generating predictions for the test set...")
model.eval()
with torch.no_grad():
    for img_path in test_images:
        # 1. Load and preprocess image (identical to BUSIDataset preprocessing, Block 7.1)
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        original_h, original_w = img_rgb.shape[:2]

        img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        img_norm = img_resized.astype(np.float32) / 255.0

        # ImageNet normalization: it must be IDENTICAL to that of BUSIDataset (Block 7.1),
        # or the model receives data with a distribution different from training
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_norm = (img_norm - mean) / std

        img_tensor = torch.tensor(img_norm, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        # 2. Predict
        # Your code here (logits -> sigmoid -> threshold > 0.5)

        # 3. Resize mask to original size
        # Your code here

        # 4. Convert to RLE and save
        rle = mask_to_rle(mask_final)
        results.append({'Id': img_path.name, 'Expected': rle})

# Create DataFrame and save
submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)
print(" submission.csv generated correctly.")